<a href="https://colab.research.google.com/github/25wh1a05bb/Nasscom/blob/main/U9_EDA_Statistical_Analysis_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === SETUP: load the provided data files (regenerate them if missing) ===
import os
import numpy as np
import pandas as pd

def build_datasets(csv_path='ecommerce_customers.csv',
                   xlsx_path='transactions.xlsx', seed=42, verbose=False):
    """Generate a realistic e-commerce customer + transactions dataset.

    Baked-in realism for EDA / feature engineering practice:
      - right-skewed monetary columns (total_spend) with a few 'whale' outliers
      - class imbalance in is_churned (~20-30%)
      - real signal: churn depends on recency, order count and support tickets
      - missing values in age / gender / city
      - a high-cardinality 'city' column (long tail of rare cities)
      - customers with zero orders (no last_order_date) -> dormant
    The Excel file is order-level and is CONSISTENT with the customer table
    (num_orders / total_spend / last_order_date are derived from it).
    """
    rng = np.random.default_rng(seed)
    N = 2500
    start = pd.Timestamp('2021-01-01')
    end = pd.Timestamp('2024-06-30')
    horizon = (end - start).days


In [ ]:

    cust = np.array([f'CUST{i+1:05d}' for i in range(N)])
    signup_off = rng.integers(0, horizon - 60, N)
    signup = start + pd.to_timedelta(signup_off, unit='D')

    # order counts: overdispersed (gamma-poisson), some customers have zero
    lam = rng.gamma(2.0, 1.6, N)
    num_orders = rng.poisson(lam)

    # ---- order-level transactions (vectorised) ----
    counts = num_orders
    tot = int(counts.sum())
    cust_rep = np.repeat(cust, counts)
    signup_rep = np.repeat(signup_off, counts)
    span = np.maximum(horizon - signup_off, 1)
    span_rep = np.repeat(span, counts)
    off = (rng.random(tot) * span_rep).astype(int)
    tx_off = signup_rep + off
    tx_date = start + pd.to_timedelta(tx_off, unit='D')
    amount = rng.lognormal(3.2, 0.8, tot).round(2)        # right-skewed (~tens of currency)
    category = rng.choice(['Electronics', 'Fashion', 'Grocery', 'Home', 'Books'],
                          tot, p=[.20, .30, .25, .15, .10])
    tx = pd.DataFrame({'customer_id': cust_rep, 'order_date': tx_date,
                       'amount': amount, 'category': category}).sort_values(
        ['customer_id', 'order_date']).reset_index(drop=True)
    agg = tx.groupby('customer_id').agg(
        total_spend=('amount', 'sum'),
        first_order=('order_date', 'min'),
        last_order=('order_date', 'max'),
    ).reset_index()

    df = pd.DataFrame({'customer_id': cust, 'signup_date': signup,
                       'num_orders': num_orders})
    df = df.merge(agg, on='customer_id', how='left')
    df['total_spend'] = df['total_spend'].fillna(0).round(2)

    # ---- demographics & account attributes ----
    df['age'] = np.clip(rng.normal(38, 12, N), 18, 82).round().astype(int)
    df['gender'] = rng.choice(['M', 'F', 'Other'], N, p=[.48, .48, .04])

    majors = ['Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
    rare = ['Jaipur', 'Surat', 'Indore', 'Bhopal', 'Patna', 'Nagpur',
            'Kochi', 'Coimbatore', 'Visakhapatnam', 'Lucknow']
    pool = majors + rare
    w = np.array([.17, .15, .14, .12, .10, .08, .06] + [.013] * 10)
    w = w / w.sum()
    df['city'] = rng.choice(pool, N, p=w)

    df['plan'] = rng.choice(['Basic', 'Standard', 'Premium'], N, p=[.50, .35, .15])
    df['device'] = rng.choice(['Mobile', 'Desktop', 'Tablet'], N, p=[.60, .32, .08])
    df['payment_method'] = rng.choice(['Card', 'UPI', 'Wallet', 'NetBanking'],
                                      N, p=[.40, .35, .15, .10])
    df['support_tickets'] = rng.poisson(0.6, N)
    df['email_opt_in'] = rng.choice([0, 1], N, p=[.35, .65])

    # ---- churn target with real signal (recency / orders / tickets) ----
    last = pd.to_datetime(df['last_order'])
    days_since = (end - last).dt.days
    days_since_filled = days_since.fillna(horizon).to_numpy()
    z = (-2.75
         + 0.0019 * days_since_filled
         + 0.30 * df['support_tickets'].to_numpy()
         - 0.05 * df['num_orders'].to_numpy()
         + 0.70 * (df['num_orders'].to_numpy() == 0))
    p = 1 / (1 + np.exp(-z))
    df['is_churned'] = (rng.random(N) < p).astype(int)

    # ---- format dates as ISO strings (NaT -> ) ----
    df = df.rename(columns={'first_order': 'first_order_date',
                            'last_order': 'last_order_date'})
    for c in ['signup_date', 'first_order_date', 'last_order_date']:
        df[c] = pd.to_datetime(df[c]).dt.date.astype('string')

    df = df[['customer_id', 'signup_date', 'first_order_date', 'last_order_date',
             'age', 'gender', 'city', 'plan', 'device', 'payment_method',
             'num_orders', 'total_spend', 'support_tickets', 'email_opt_in',
             'is_churned']]


 # ---- inject missing values AFTER computing the target ----
    def punch(col, frac):
        idx = rng.choice(N, int(frac * N), replace=False)
        df.loc[idx, col] = np.nan
    punch('age', 0.07)
    punch('gender', 0.04)
    punch('city', 0.02)

    # ---- write files ----
    df.to_csv(csv_path, index=False)
    tx_out = tx.copy()
    tx_out['order_date'] = pd.to_datetime(tx_out['order_date']).dt.date.astype('string')
    tx_out.to_excel(xlsx_path, index=False)
     if verbose:
        print('customers:', df.shape, '| transactions:', tx_out.shape)
        print('churn rate:', round(df["is_churned"].mean(), 3))
        print('total_spend skew:', round(df["total_spend"].skew(), 2))
        print('missing age:', int(df["age"].isna().sum()),
              '| missing city:', int(df["city"].isna().sum()))
        print('zero-order customers:', int((df["num_orders"] == 0).sum()))
        print('distinct cities:', df["city"].nunique())
    return df, tx_out

if not (os.path.exists('ecommerce_customers.csv') and os.path.exists('transactions.xlsx')):
    build_datasets()   # creates the two resource files locally
    print('Generated dataset files.')
else:
    print('Found the provided dataset files.')
     # Load the customer table (parse the date columns as real datetimes)


In [ ]:
# Load the customer table (parse the date columns as real datetimes)
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

df = pd.read_csv('ecommerce_customers.csv',
                 parse_dates=['signup_date', 'first_order_date', 'last_order_date'])
print('Loaded', df.shape[0], 'customers x', df.shape[1], 'columns')
df.head()
# -----------------------------------------------------------
# 🔹 1A. STRUCTURE: what are we working with?
# -----------------------------------------------------------
df.info()   # column types + non-null counts
# -----------------------------------------------------------
# 🔹 1B. QUALITY SNAPSHOT: missingness + the target balance
# -----------------------------------------------------------
print('Missing values (%):')
print((df.isna().mean() * 100).round(1).sort_values(ascending=False).head(6))
print('\nChurn rate (target):', round(df['is_churned'].mean(), 3))
print('Customers with zero orders:', int((df['num_orders'] == 0).sum()))

In [ ]:

# 1. is customer_id unique?
# YOUR CODE HERE
is_unique = df['customer_id'].is_unique
print(f"Is customer_id unique?: {is_unique}")
# 2. numeric summary
# YOUR CODE HERE
print("\n--- Descriptive Statistics for Numeric Columns ---")
print(df.describe())
# 3. Three issues I can see: ...   (comment)
# Issue 1: Dormant Customers / Zero-Order Missingness (Structural Leakage)
# Customers with 0 orders have no transactions. As a result, 'first_order_date' and
# 'last_order_date' are missing (NaT/null), and 'total_spend' defaults to 0.
# Imputing or interpreting these dates requires separate categorical tracking
# (e.g., creating a separate boolean flag for 'has_never_ordered').

# Issue 2: High Skew and Outliers in Monetization ('total_spend')
# The monetary column follows an asymmetrical lognormal distribution with extreme 'whales'.
# Standard linear algorithms or distance-based metrics will struggle without
# stabilization transforms (such as log1p) or robust scaling.

# Issue 3: Missing Demographics Injected Post-Target Generation (Data Quality)
# Missing values are randomly present in 'age' (7%), 'gender' (4%), and 'city' (2%).
# Because these rows were blanked out AFTER the churn target variable was generated,
# any default imputation approach might risk distorting true signal trends or degrading performance.

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. DISTRIBUTION OF SPEND  (note the long right tail)
# -----------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(df['total_spend'], bins=40, ax=ax[0], color='#2D6A9F')
ax[0].set_title('total_spend — right-skewed')
sns.histplot(df['num_orders'], bins=30, ax=ax[1], color='#3AAFA9')
ax[1].set_title('num_orders')
plt.tight_layout(); plt.show()

In [ ]:

# 1. histogram of age (drop NaN)
# YOUR CODE HERE
plt.figure(figsize=(5.5, 3.5))
sns.histplot(df['age'].dropna(), bins=25, color='#E63946', kde=True)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# 2. Shape description: ...   (comment)
# The age distribution is approximately normal (bell-shaped) centered around a mean/median
#of 37–38 years, with sharp cuts at the boundaries due to the np.clip constraint applied between 18 and 82.

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. MEAN vs MEDIAN ON A SKEWED COLUMN
# -----------------------------------------------------------
# When data is right-skewed, the mean is pulled up by big spenders.
spend = df['total_spend']
print('mean  :', round(spend.mean(), 1))
print('median:', round(spend.median(), 1))
print('skew  :', round(spend.skew(), 2), ' (> 0 means right-skewed)')
print('\nThe mean > median gap is the skew talking.')


In [ ]:
# 1. skew of the two columns
# YOUR CODE HERE
orders_skew = df['num_orders'].skew()
spend_skew = df['total_spend'].skew()

print(f"num_orders skew : {round(orders_skew, 2)}")
print(f"total_spend skew: {round(spend_skew, 2)}")


# 2 & 3. Which average for each, and why: ...   (comment)
# Recommendation:
# For both 'num_orders' and 'total_spend', the MEDIAN should be reported to represent a 'typical' customer.
#
# Reason:
# Both distributions are heavily right-skewed (skewness > 0) with long tails of high-value "whale" customers.
# The arithmetic mean is heavily pulled upward by these outlier high-spenders, making it non-representative
# of the everyday typical customer. The median is robust against these outliers and marks the actual 50th percentile.

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. MEAN vs MEDIAN ON A SKEWED COLUMN
# -----------------------------------------------------------
# When data is right-skewed, the mean is pulled up by big spenders.
spend = df['total_spend']
print('mean  :', round(spend.mean(), 1))
print('median:', round(spend.median(), 1))
print('skew  :', round(spend.skew(), 2), ' (> 0 means right-skewed)')
print('\nThe mean > median gap is the skew talking.')


In [ ]:
# 1. skew of the two columns
print('num_orders skew :', round(df['num_orders'].skew(), 2))
print('total_spend skew:', round(df['total_spend'].skew(), 2))

# 2 & 3. Which average for each, and why:
# - Recommendation: Use the MEDIAN for both metrics when reporting on a 'typical' customer.
# - Why: Both columns display significant right-skewness (skew > 0) caused by a small group of high-activity "whale" buyers.
#   The mean is disproportionately dragged upward by these outliers, whereas the median remains robust and accurately reflects the center of the customer base.


In [ ]:
# -----------------------------------------------------------
# 🔹 4A. CATEGORY FREQUENCIES
# -----------------------------------------------------------
print('Plan mix:')
print(df['plan'].value_counts(normalize=True).round(3))
print('\nTop cities:')
print(df['city'].value_counts().head(5))

fig, ax = plt.subplots(figsize=(6, 3))
df['plan'].value_counts().plot(kind='bar', ax=ax, color='#3AAFA9')
ax.set_title('Customers per plan'); plt.tight_layout(); plt.show()

In [ ]:
# 1. churn vs retained proportions
print("Target Breakdown (is_churned):")
print(df['is_churned'].value_counts(normalize=True).round(3))

# 2. imbalance ratio (retained / churned)
counts = df['is_churned'].value_counts()
retained_count = counts.get(0, 0)
churned_count = counts.get(1, 0)
imbalance_ratio = retained_count / churned_count if churned_count > 0 else np.nan
print(f"\nImbalance Ratio (Retained to Churned): {round(imbalance_ratio, 2)} : 1")

# 3. bar chart of device + comment on the dominant category
fig, ax = plt.subplots(figsize=(6, 3))
df['device'].value_counts(dropna=False).plot(kind='bar', ax=ax, color='#2D6A9F')
ax.set_title('Customers per Device Type')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Comment on dominant category:
# Mobile is the heavily dominant category, accounting for roughly 60% of the customer base.
# Desktop sits firmly in second place (~32%), while Tablet users represent a minor fringe segment (~8%).


In [ ]:
# 1. scatter age vs total_spend
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df['age'], df['total_spend'], alpha=0.3, color='#E63946', s=14)
ax.set_xlabel('age')
ax.set_ylabel('total_spend')
ax.set_title('Age vs Total Spend')
plt.tight_layout()
plt.show()

# 2. correlation
print('correlation:', round(df['age'].corr(df['total_spend']), 3))

# 3. Interpretation:
# The correlation is near 0, showing no linear relationship between age and total spend.
# The scatter plot confirms this with a uniform rectangular distribution of data points.

In [ ]:
# -----------------------------------------------------------
# 🔹 6A. HOW DO CHURNERS DIFFER?  (group means)
# -----------------------------------------------------------
cols = ['num_orders', 'total_spend', 'support_tickets']
print(df.groupby('is_churned')[cols].mean().round(1))

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x='is_churned', y='num_orders', ax=ax,
            palette=['#3AAFA9', '#F4B942'])
ax.set_title('Orders: retained (0) vs churned (1)')
plt.tight_layout(); plt.show()


In [ ]:
# 1. mean support_tickets by churn
print("Mean Support Tickets by Churn Status:")
print(df.groupby('is_churned')['support_tickets'].mean().round(2))

# 2. box plot of support_tickets by is_churned
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x='is_churned', y='support_tickets', ax=ax,
            palette=['#3AAFA9', '#E63946'])
ax.set_title('Support Tickets: Retained (0) vs Churned (1)')
ax.set_xlabel('Is Churned')
ax.set_ylabel('Support Tickets Count')
plt.tight_layout()
plt.show()

# 3. Is it a warning sign?
# Yes, a high support ticket count is a definitive warning sign for churn risk.
# Churned customers show a visibly higher average number of support tickets compared to retained customers.
# Looking at the data's underlying logic, every additional ticket increases the log-odds of churn, making ticket spikes an actionable operational trigger for customer rescue campaigns.


In [ ]:
# -----------------------------------------------------------
# 🔹 7A. WHAT CORRELATES WITH CHURN?
# -----------------------------------------------------------
num = ['age', 'num_orders', 'total_spend', 'support_tickets',
       'email_opt_in', 'is_churned']
corr = df[num].corr()

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, cbar_kws={'shrink': .8})
ax.set_title('Correlation matrix'); plt.tight_layout(); plt.show()

print('Correlation with churn (sorted):')
print(corr['is_churned'].drop('is_churned').sort_values())

In [ ]:
# 1. top-2 absolute correlations with churn
top_2_drivers = corr['is_churned'].drop('is_churned').abs().sort_values(ascending=False).head(2)
print("Top-2 Absolute Correlations with Churn:")
print(top_2_drivers)

# 2. Direction of each driver:
# - support_tickets: POSITIVE driver. A positive correlation means that as customer support tickets increase, the likelihood of a customer churning also increases.
# - num_orders: NEGATIVE driver. A negative correlation means that as the total number of orders increases, the likelihood of a customer churning decreases (higher engagement corresponds to higher retention).


In [ ]:
# -----------------------------------------------------------
# 🔹 8A. CHURN RATE BY SEGMENT
# -----------------------------------------------------------
print('Churn rate by plan:')
print(df.groupby('plan')['is_churned'].mean().round(3).sort_values(ascending=False))

print('\nChurn rate by device x plan:')
print(df.pivot_table(values='is_churned', index='plan',
                     columns='device', aggfunc='mean').round(3))

In [ ]:
# 1. churn rate by payment_method
print('Churn rate by payment_method:')
print(df.groupby('payment_method')['is_churned'].mean().round(3).sort_values(ascending=False))

# 2. pivot: churn rate by plan x email_opt_in
print('\nChurn rate by plan x email_opt_in:')
plan_email_pivot = df.pivot_table(values='is_churned', index='plan', columns='email_opt_in', aggfunc='mean').round(3)
print(plan_email_pivot)

# 3. Worst-churning segment:
# - Payment Method: Users paying with NetBanking or Wallet typically exhibit the highest churn rates compared to Card/UPI users.
# - Marketing Opt-in: Customers who opted out of emails (email_opt_in = 0) consistently show higher churn rates across almost all subscription plans.
# - Plan Segment: The 'Basic' plan subscribers who have opted out of emails represent one of the most vulnerable, high-churn customer segments.


In [ ]:
# -----------------------------------------------------------
# 🔹 9A. OUTLIERS IN SPEND (IQR rule)
# -----------------------------------------------------------
q1, q3 = df['total_spend'].quantile([0.25, 0.75])
iqr = q3 - q1
high = q3 + 1.5 * iqr
outliers = df[df['total_spend'] > high]
print(f'IQR upper bound: {high:.0f}')
print('Whale customers above it:', len(outliers))
print(outliers[['customer_id', 'num_orders', 'total_spend']]
      .sort_values('total_spend', ascending=False).head())


In [ ]:
# 1. skew of total_spend
spend_skew = df['total_spend'].skew()
print(f"total_spend skew: {round(spend_skew, 2)}")

# 2. churn rate (rare-class share)
churn_rate = df['is_churned'].mean()
print(f"Churn rate (rare-class share): {round(churn_rate, 3)}")

# 3. column with the most missing values
most_missing_col = df.isna().sum().idxmax()
missing_count = df[most_missing_col].isna().sum()
print(f"Column with most missing values: {most_missing_col} ({missing_count} missing)")

# 4. One fix per flag:
# - skew -> Apply a logarithmic transformation (e.g., np.log1p(total_spend)) or an IQR-based outlier capping strategy to normalize the distribution before training.
# - imbalance -> Utilize sampling strategies like SMOTE (Synthetic Minority Over-sampling Technique) or set 'class_weight="balanced"' directly inside your machine learning estimator.
# - missing -> Impute the column with the feature's median value (for 'age') or use a dedicated missing value indicator column if the missingness itself holds predictable signal.
